# B.3: Ablate-IMV with simulated data

Simulate a binary outcome and compare a neural network with two hidden layers against an ablated version with its second hidden layer removed. Both variants are fitted from scratch on the same training observations, and `AblationIMV` compares their probabilities on the same test observations.

Run the cell below from any working directory. The base `imvpy` installation supplies the required NumPy, pandas, and scikit-learn dependencies; no data download is needed.

In [1]:
# Install once: python -m pip install imvpy
import numpy as np
import pandas as pd
from imvpy import AblationIMV
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

# 1. Simulate and split a binary outcome.
rng = np.random.default_rng(42)
X = rng.normal(size=(2000, 3))
p = 1 / (1 + np.exp(-(2 * X[:, 0] + X[:, 1])))
y = rng.binomial(1, p)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# 2. Fit the full and ablated architectures.
predictions = {}
variants = {"Full": (8, 8), "Ablated": (8,)}
for name, layers in variants.items():
    model = MLPClassifier(
        hidden_layer_sizes=layers, solver="lbfgs",
        alpha=1, max_iter=2000, random_state=42
    ).fit(X_train, y_train)
    predictions[name] = pd.DataFrame({
        "True Label": y_test,
        "Positive Probability": model.predict_proba(X_test)[:, 1],
    })

# 3. Compare models on the same test observations.
matrix = AblationIMV.calculate_imv_matrix(predictions)
print(matrix.round(3))

          Full  Ablated
Full     0.000   -0.014
Ablated  0.014    0.000


Each matrix entry evaluates the row model relative to the column model. The negative `Full`–`Ablated` entry indicates that the additional layer does not improve held-out fit in this simulation. This is an architecture ablation with retraining; the directional matrix need not be symmetric.